# Results

In [ ]:
from collections import defaultdict
from ast import literal_eval  # to convert string list ↦ python list
import random

# CORE FUNCTION
def sample_next_token(prob_dicts, sequence, n):
	prefix = ()
	# Get the current prefix (last n-1 words)
	if n > 1:
		prefix = tuple(sequence[-(n-1):])

	# In the probability dictionary, get possible next tokens for the given prefix
	next_tokens = prob_dicts[n].get(prefix, {})
	# If no next tokens found, back off to (n-1)-gram model
	if next_tokens == {}:
		return sample_next_token(prob_dicts, sequence, n-1)				# RECURSION: try with n-1-gram
	
	words = list(next_tokens.keys())
	probs = list(next_tokens.values())
	
	next_token = random.choices(words, weights=probs, k=1)[0]
	print(f"Current n: {n}, Current prefix: {prefix}, Next token: {next_token} ")
	return next_token

## Task 1: Generate Parragraphs

In [2]:
# TASK 1
def generative_ngram(n, mt, prob_json, nr_outputs=1):
	# Create a list for probability dictionaries
	prob_dicts = [{}]

	for k in range(1, n+1):
		prob_dict = {}
		for prefix_str, next_tokens in prob_json[k].items():
			prefix = literal_eval(prefix_str)
			prob_dict[prefix] = {}
			for token, prob in next_tokens.items():
				prob_dict[prefix][token] = prob
		prob_dicts.append(prob_dict)

	# Initialize output list
	output = [""] * nr_outputs

	for i in range(nr_outputs):
		print("NEW OUTPUT IN PROGRESS")
		if mt == "word":
			# Generate a sentence starting from <s> and ending at </s> based on the probability dictionary
			sequence = ["<s>"]
			
			# As long as the end symbol </s> is not reached, keep sampling the next word
			while sequence[-1] != "</s>":
				next_word = sample_next_token(prob_dicts, sequence, n)
				sequence.append(next_word)
			
			output[i] = " ".join(sequence)		# Convert list of generated words to a single string using space as separator

		else:
			# Generate a character sequence ending at ".", "!" or "?" based on the probability dictionary
			sequence = [""]
			# As long as none of the end symbols is reached, keep sampling the next character
			while sequence[-1] != "." and sequence[-1] != "!" and sequence[-1] != "?":
				next_char = sample_next_token(prob_dicts, sequence, n)
				sequence.append(next_char)
			
			output[i] = "".join(sequence)		# Convert list of generated characters to a single string

	return output

## Task 2: Complete Sentences

In [1]:
# TASK 2
def complete_sentence(sentence, n, prob_json):
	# Create a list for probability dictionaries
	prob_dicts = [{}]

	for k in range(1, n+1):
		prob_dict = {}
		for prefix_str, next_tokens in prob_json[k].items():
			prefix = literal_eval(prefix_str)
			prob_dict[prefix] = {}
			for token, prob in next_tokens.items():
				prob_dict[prefix][token] = prob
		prob_dicts.append(prob_dict)

	# Initialize output sequence with the given sentence start
	sequence = sentence.copy()
			
	# As long as the end symbol ".", "!" or "?" is not reached, keep sampling the next word
	while sequence[-1] != "." and sequence[-1] != "!" and sequence[-1] != "?":
		next_word = sample_next_token(prob_dicts, sequence, n)
		sequence.append(next_word)
		
	output = " ".join(sequence)		# Convert list of generated words to a single string using space as separator
	return output

## Model based on Testing Data

In [3]:
import os
import json

N = {5}													# Fallback strategy: if a prefix does not exist in N-gram dictionary, back off to N-1-gram model
modeltype = {"word"}									# "word", "character"
categories = {"business", "entertainment", "politics", "sport", "tech", "all"}	# "business", "entertainment", "politics", "sport", "tech", "all"

sourcefolder = f"./../Probability_NGrams/"
targetfolder = f"./../Task1_GeneratedSequences/"

## View Results